# IS2SMGPSIT-V1 interactive maps

**Summary**: Interactive (pan/zoom/hover) polar maps of the daily IS2SMGPSIT-V1 fused sea ice thickness product, rendered with GeoViews/hvPlot in a true North Polar Stereographic projection. Includes a winter-mean map slider across the seven growth seasons and a monthly map slider through the most recent season.

**Author**: Alek Petty  
**Version history**: Version 1 (08/2026)


In [1]:
### Import notebook dependencies

import xarray as xr
import numpy as np
import pandas as pd
import warnings

from utils.read_data_utils import read_is2smgpsitv1_zarr
from utils.plotting_utils import interactiveArcticMaps, compute_gridcell_winter_means

import cartopy.crs as ccrs
import hvplot.xarray
import holoviews as hv

warnings.filterwarnings('ignore')


## Load the IS2SMGPSIT-V1 data

Load the daily fused product from the S3 Zarr store (cached locally after first download) and resample to calendar-monthly means for the interactive displays.


In [2]:
IS2_SMOS_SMAP = read_is2smgpsitv1_zarr(load_cache=True)

# Static 2D lon/lat fields (collapse the redundant time dimension if reading
# directly from S3, where they are stored per time step)
lon2d = IS2_SMOS_SMAP.longitude
lat2d = IS2_SMOS_SMAP.latitude
if 'time' in lon2d.dims:
    lon2d = lon2d.isel(time=0, drop=True)
    lat2d = lat2d.isel(time=0, drop=True)

IS2_monthly = IS2_SMOS_SMAP.ice_thickness.resample(time='1MS').mean(keep_attrs=True)

# resample().mean() drops the 2D longitude/latitude coordinates, which the
# interactive quadmesh plots need to project the data; re-attach them.
IS2_monthly = IS2_monthly.assign_coords(longitude=lon2d, latitude=lat2d)
IS2_monthly.attrs['long_name'] = 'Sea ice thickness'
IS2_monthly.attrs['units'] = 'm'
print('monthly time steps:', IS2_monthly.sizes['time'])


Loading IS2-SMOS-SMAP (is2smsitgp) Zarr from local cache
cache_path: ./data/cache/GPSat_multivar_20181101-20250430.zarr


monthly time steps: 78


## Winter mean thickness (interactive slider)

November–April grid-cell mean thickness for the seven growth seasons. Use the slider to step through winters; hover for grid-cell values, and use the toolbar to pan/zoom.


In [3]:
years = [x for x in range(2018, 2024 + 1)]

# Compute winter means without the 2D lon/lat coords (concatenating winters
# would otherwise broadcast them along time), then re-attach them for plotting.
thickness_winter_means = compute_gridcell_winter_means(
    IS2_monthly.drop_vars(['longitude', 'latitude']), years=years,
)
thickness_winter_means = thickness_winter_means.assign_coords(longitude=lon2d, latitude=lat2d)

interactiveArcticMaps(
    thickness_winter_means,
    clabel='Sea ice thickness (m)',
    cmap='viridis',
    vmin=0,
    vmax=5,
    frame_width=500,
    pixel_ratio=0.5,
)


:HoloMap   [time]
   :Overlay
      .Image.I     :Image   [longitude,latitude]   (Sea ice thickness)
      .Coastline.I :Feature   [Longitude,Latitude]

## Monthly thickness through the most recent season (interactive slider)

Monthly mean thickness for the 2024–2025 growth season (September 2024 through April 2025).


In [4]:
latest_season = IS2_monthly.sel(time=slice('2024-09-01', '2025-04-30'))

interactiveArcticMaps(
    latest_season,
    clabel='Sea ice thickness (m)',
    cmap='viridis',
    vmin=0,
    vmax=5,
    frame_width=500,
    pixel_ratio=0.5,
)


:HoloMap   [time]
   :Overlay
      .Image.I     :Image   [longitude,latitude]   (Sea ice thickness)
      .Coastline.I :Feature   [Longitude,Latitude]

## Notes

- Maps are rendered with `interactiveArcticMaps` (GeoViews/hvPlot `quadmesh`, rasterized) in a North Polar Stereographic projection, so the full domain including the pole displays correctly (no web-Mercator cutoff).
- Displays use monthly or winter means rather than the full daily record, rendered at a reduced raster resolution (`pixel_ratio=0.5`), to keep the embedded page size reasonable; the underlying product is daily at 25 km.
- For static publication-quality versions of these maps see notebook `11e`.
